# Capítulo 16 — Tu escena (sin guion)

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

Hasta aquí, cada paso venía dado: una escena fallaba, descubríamos la necesidad y construíamos la pieza. **Ahora la historia sigue, pero el guion lo escribes tú.**

| | |
|---|---|
| **Formato** | Por parejas · 90 min de trabajo + 5 min de presentación por pareja |
| **Material** | Todo lo construido en los capítulos 0–15, la Parte I del documento como manual y el cuaderno de deudas |
| **Regla de oro** | Cada componente que añadas debe responder a una **necesidad** que puedas contar como una escena. Si no sabes qué escena lo justifica, sobra |

In [ ]:
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()
taximetro_historico()

## Elige una escena

### A · El mostrador
TiendaNova abre **12 tiendas físicas**. Los dependientes tramitarán devoluciones en el mostrador con un agente interno en una tablet:
- Un dependiente puede aprobar reembolsos de hasta 200 €; el encargado, hasta 1.000 €.
- Cada tienda solo puede ver **sus** devoluciones.
- El cliente recibe un aviso por su canal preferido cuando se aprueba el reembolso.

*Piezas que pondrás a prueba:* identidad con roles, Policy con *claims*, aislamiento por tienda (multi-tenant), memoria.

### B · El marketplace
TiendaNova incorpora **vendedores externos** que tienen sus propios agentes de atención, hechos con otros frameworks:
- Nova debe derivarles las consultas sobre sus productos.
- Los vendedores no pueden ver datos de clientes que no sean suyos.
- Dirección quiere saber qué vendedor genera más incidencias.

*Piezas que pondrás a prueba:* A2A, Registry, el Gateway como frontera, contexto mínimo, OBO, analítica sobre Iceberg.

### C · Black Friday
Durante 72 horas el tráfico se multiplica por 20:
- Finanzas fija un **presupuesto máximo de inferencia**.
- El 15 % de las conversaciones serán sobre retrasos.
- Si el sistema se satura, el cliente no puede quedarse sin respuesta.

*Piezas que pondrás a prueba:* cuotas y *throttling*, inference profiles, TPM por canal, *prompt caching*, modelos más pequeños para pasos simples, degradación elegante, evaluación online con muestreo.

In [ ]:
ESCENA = "A"   # ← "A", "B" o "C"
state_set(ESCENA_CAP16=ESCENA)

## Entregables

1. **La historia de tu escena:** de tres a cinco momentos, cada uno con **qué falla → qué necesidad revela → qué pieza la resuelve → por qué**.
2. **Arquitectura:** un diagrama Mermaid que parta de la de Nova Assist y marque lo nuevo.
3. **Dos piezas implementadas** sobre lo que ya tienes. Por ejemplo: una política Cedar y una herramienta nueva; o un límite de TPM y una degradación elegante.
4. **Tres casos de evaluación nuevos**, uno de ellos adversarial, añadidos a `eval_casos` en Iceberg y ejecutados.
5. **El taxímetro:** coste por conversación antes y después de tus cambios.
6. **Deudas honestas:** lo que queda pendiente, anotado en el cuaderno con su motivo.

## 1. La historia de tu escena

In [ ]:
HISTORIA = [
    # (qué falla, necesidad, pieza, por qué)
    ("", "", "", ""),
    ("", "", "", ""),
    ("", "", "", ""),
]
for i, (falla, necesidad, pieza, porque) in enumerate(HISTORIA, 1):
    print(f"{i}. 🎬 {falla}\n   → necesidad: {necesidad}\n   → pieza: {pieza}\n   → por qué: {porque}\n")

## 2. Arquitectura

Edita este diagrama (parte de Nova Assist) y marca lo nuevo con `:::nuevo`.

```mermaid
flowchart TB
    classDef nuevo stroke-width:3px,stroke-dasharray: 5 5
    usuario["¿Quién habla?"] --> rt["Runtime: NovaAgent"]
    rt --> mem["Memory"]
    rt --> gw["Gateway · Policy · Guardrails"]
    gw --> lped["Lambda pedidos → Iceberg"]
    gw --> lpol["buscar_politicas → S3 Vectors"]
    nuevo1["Tu pieza nueva"]:::nuevo
```

## 3. Dos piezas implementadas

Trabaja en celdas nuevas debajo de esta. Reutiliza todo lo que necesites: `fijar_dependencia`, `escribir_tabla`, `athena`, los `%%writefile` de los capítulos anteriores o `agentcore add …`.

In [ ]:
# Pieza 1

In [ ]:
# Pieza 2

## 4. Tres casos de evaluación nuevos (uno adversarial)

In [ ]:
NUEVOS_CASOS = [
    {"caso_id": "T01", "categoria": "tu_escena", "cliente": "C-001", "prompt": "",
     "herramientas_esperadas": "", "respuesta_esperada": "", "documento_esperado": ""},
    {"caso_id": "T02", "categoria": "tu_escena", "cliente": "C-001", "prompt": "",
     "herramientas_esperadas": "", "respuesta_esperada": "", "documento_esperado": ""},
    {"caso_id": "T03", "categoria": "adversarial", "cliente": "C-001", "prompt": "",
     "herramientas_esperadas": "", "respuesta_esperada": "", "documento_esperado": ""},
]
assert all(c["prompt"] for c in NUEVOS_CASOS), "Completa los prompts"
escribir_tabla("eval_casos", NUEVOS_CASOS, modo="append")
for c in NUEVOS_CASOS:
    invocar(c["prompt"], token=token_cliente(c["cliente"]))

## 5. El taxímetro antes y después

Ejecuta la misma conversación representativa de tu escena **antes** y **después** de tus cambios, en sesiones distintas, y compáralas.

In [ ]:
CONVERSACION = ["", "", ""]   # ← mensajes representativos de tu escena
sesion = new_session_id()
for mensaje in CONVERSACION:
    invocar(mensaje, session_id=sesion, token=token_cliente("C-001"))
taximetro()

## 6. Deudas honestas

In [ ]:
# Ejemplo: deuda("T1", "abierta", 16, "motivo", descripcion="Qué queda pendiente")
cuaderno()

<details>
<summary>💡 Pistas (ábrelas solo si te atascas)</summary>

**A · El mostrador**
- Los roles viajan como *claims* del token: piensa en un pool de Cognito con grupos y en `principal.getTag(...)` en Cedar (capítulos 9 y 10).
- «Cada tienda solo ve lo suyo» es multi-tenant: revisa la tabla de aislamiento por capa (doc §7).
- El aviso al cliente necesita su canal preferido: está en la memoria a largo plazo (capítulo 5). ¿Quién debe poder leerla?

**B · El marketplace**
- Un agente externo es otro departamento: A2A, a través del Gateway, y publicado en Registry (capítulo 14, §3.4 y §4.4).
- Contexto mínimo: ¿qué datos del cliente necesita de verdad el vendedor? (capítulo 14).
- La analítica por vendedor es una consulta predefinida sobre Iceberg, igual que las del analista.

**C · Black Friday**
- Vuelve al capítulo 1: inference profile, reintentos adaptativos y degradación elegante.
- Límites de TPM por canal en el Gateway (doc §3.3) y *prompt caching* con el prefijo estable (capítulo 6).
- ¿Qué pasos podría hacer un modelo más pequeño? Mídelo con la evaluación del capítulo 12 antes de cambiarlo.
</details>

## Rúbrica (autoevaluación y evaluación entre parejas)

| Criterio | 1 · Inicial | 2 · En desarrollo | 3 · Competente | 4 · Excelente |
|---|---|---|---|---|
| **Necesidad → pieza** | Lista de componentes sin motivo | Algunos motivos | Cada pieza responde a una escena | Las escenas encadenan una historia, como en el curso |
| **Hilo de producción** | No aparece | Mencionado | Se razona qué se rompe en local | Se razona y se mide |
| **Seguridad** | Sin considerar | Identidad *o* políticas | Mínimo privilegio, identidad e inyección | Incluye un caso adversarial que falla sin la pieza y pasa con ella |
| **Coste** | Sin medir | Estimado de palabra | Medido con el taxímetro | Medido antes y después, con una decisión justificada |
| **Evaluación** | Sin casos | Casos sin ejecutar | 3 casos ejecutados | Casos en Iceberg, incluido uno adversarial, con resultado interpretado |
| **Honestidad** | Todo «resuelto» | Deudas vagas | Deudas concretas | Deudas concretas con capítulo o pieza que las pagaría |

## Presentación (5 minutos)

Cuéntalo **como una historia**: escena → descubrimiento → pieza → lo que queda pendiente. Sin diapositivas de componentes: si el grupo no entiende por qué existe una pieza, la historia no está bien contada.

In [ ]:
guardar_partida(16)

---
🎓 **Fin del curso.** Antes de irte, ejecuta el capítulo 15 (limpieza) si no lo has hecho.